# Adversarial TTS - Class-Based Architecture (Google Colab)

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles argument parsing, model loading, and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

This separation allows you to re-run logging/visualization without re-running optimization.

## Setup
Run the cell below to clone the repository and install dependencies.

In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

## 1. Configuration & Imports

In [ ]:
%cd StyleTTS2

import torch
import argparse
import nltk
nltk.download('punkt_tab')

# Import class-based modules
from Trainer.EnvironmentLoader import EnvironmentLoader
from Trainer.AdversarialTrainer import AdversarialTrainer
from Trainer.RunLogger import RunLogger

# Import Pymoo components
from Optimizer.pymoo_optimizer import PymooOptimizer
from pymoo.algorithms.moo.nsga2 import NSGA2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

# =============================================================================
# Configuration - Edit these values directly
# =============================================================================
args = argparse.Namespace(
    # Text parameters
    ground_truth_text="I think the NFL is lame and boring",
    target_text="The Seattle Seahawks are the best Team in the world",
    
    # Optimization parameters
    loop_count=1,
    num_generations=4,
    pop_size=4,
    iv_scalar=0.5,
    size_per_phoneme=1,
    batch_size=-1,  # -1 for full batch
    
    # Flags
    notify=False,
    subspace_optimization=False,
    multi_gpu=False,
    
    # Mode and objectives
    mode="TARGETED",  # TARGETED, UNTARGETED, etc.
    ACTIVE_OBJECTIVES=["PESQ", "WER_GT"],
    thresholds=["PESQ=0.3", "WER_GT=0.5"],
)

## 2. Initialize Environment

Multi-GPU is automatically configured during initialization if `--multi_gpu` flag is set.

In [ ]:
# Initialize environment using EnvironmentLoader
loader = EnvironmentLoader(device)
config_data, model_data, audio_data, objective_manager = loader.initialize(args)

print("\nEnvironment initialized successfully!")

## 3. Run Optimization

The trainer only runs optimization and returns results. Logging is done separately.

In [ ]:
# Create trainer
trainer = AdversarialTrainer(config_data, model_data, audio_data, objective_manager, device)

# Run optimization loops
for loop_iteration in range(config_data.loop_count):
    print(f"\n[Loop {loop_iteration + 1}/{config_data.loop_count}] Starting optimization loop...")

    # Initialize fresh optimizer for this cycle
    optimizer = PymooOptimizer(
        bounds=(0, 1),
        algorithm=NSGA2,
        algo_params={"pop_size": config_data.pop_size},
        num_objectives=len(config_data.active_objectives),
        solution_shape=(audio_data.input_lengths.detach().cpu().item(), config_data.size_per_phoneme),
    )

    fitness_data, generation_count, elapsed_time_total = trainer.run_full_iteration(optimizer)

    # Log Results
    logger = RunLogger(optimizer, config_data, model_data, audio_data, fitness_data, generation_count, elapsed_time_total, device)
    logger.finalize_run()

    print(f"Results saved to: {logger.folder_path}")

## 4. Re-generate Visualizations (Optional)

You can regenerate graphs with different settings without re-running optimization.

In [ ]:
from Trainer.GraphPlotter import GraphPlotter

# Create a new plotter (you could customize settings here)
plotter = GraphPlotter(
    folder_path=logger.folder_path,
    active_objectives=config_data.active_objectives
)

# Regenerate all visualizations
plotter.generate_all_visualizations(fitness_data)
print("Visualizations regenerated!")

## 5. Download Results (Optional)

In [ ]:
extract = False  # Set to True to download results

if extract:
    !zip -r colab_results.zip /content/StyleTTS2/output/

    from google.colab import files
    files.download('colab_results.zip')